# 03 — Visualisasi Data (Grafik PSAJ)
**PSAJ Statistika | Bab 4.3 Laporan**

Notebook ini menghasilkan grafik-grafik wajib PSAJ dalam **resolusi tinggi (300 dpi)**:
- Histogram
- Poligon Frekuensi
- Ogif Positif (Kurang Dari)
- Ogif Negatif (Lebih Dari)
- Scatter Plot X vs Y (preview)

---
**Input:** `outputs/merged_2025.csv` + `outputs/tables/tabel_frekuensi.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import math, os

os.makedirs('outputs/figures', exist_ok=True)

# Style global untuk grafik berkualitas tinggi
plt.rcParams.update({
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'font.family'    : 'sans-serif',
    'axes.titlesize' : 13,
    'axes.titleweight': 'bold',
    'axes.labelsize' : 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.grid'      : True,
    'grid.alpha'     : 0.3,
})

# Load data
df = pd.read_csv('outputs/merged_2025.csv')
X  = df['SMA_Plus_Pct'].values
Y  = df['TPT_Pct'].values
n  = len(X)

# Rebuild tabel frekuensi
x_min = X.min(); x_max = X.max()
R     = x_max - x_min
k     = math.ceil(1 + 3.3 * math.log10(n))
p     = math.ceil(R / k)
batas_bawah_awal = math.floor(x_min)
bins  = [batas_bawah_awal + i * p for i in range(k + 1)]
freq, _ = np.histogram(X, bins=bins)

tepi_bawah   = [bins[i] - 0.5 for i in range(k)]
tepi_atas    = [bins[i+1] - 0.5 for i in range(k)]
titik_tengah = [(tepi_bawah[i] + tepi_atas[i]) / 2 for i in range(k)]
fk_kurang    = np.cumsum(freq)
fk_lebih     = n - np.cumsum(freq) + freq

print(f'Data: n={n}, k={k} kelas, p={p}')
print('Frekuensi tiap kelas:', freq)

## Grafik 1: Histogram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bar_colors = plt.cm.Blues(np.linspace(0.4, 0.85, k))
ax.bar(
    [bins[i] + p/2 for i in range(k)],  # posisi tengah bar
    freq,
    width=p * 0.95,
    color=bar_colors,
    edgecolor='navy',
    linewidth=0.8
)

# Label frekuensi di atas bar
for i in range(k):
    ax.text(bins[i] + p/2, freq[i] + 0.1, str(freq[i]),
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks([bins[i] for i in range(k+1)])
ax.set_xlabel('Persentase Penduduk Usia 25+ dengan Pendidikan SMA ke Atas (%)')
ax.set_ylabel('Frekuensi (Jumlah Provinsi)')
ax.set_title('Histogram: Distribusi Persentase Lulusan SMA+\n38 Provinsi di Indonesia (2025)')
ax.set_ylim(0, max(freq) + 2)

plt.tight_layout()
plt.savefig('outputs/figures/01_histogram.png', bbox_inches='tight')
plt.show()
print('Tersimpan: outputs/figures/01_histogram.png')

## Grafik 2: Poligon Frekuensi

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Tambah titik nol di awal dan akhir untuk menutup poligon
tt_extended = [titik_tengah[0] - p] + list(titik_tengah) + [titik_tengah[-1] + p]
fr_extended = [0] + list(freq) + [0]

ax.plot(tt_extended, fr_extended, 'o-', color='darkblue',
        linewidth=2, markersize=7, markerfacecolor='steelblue', label='Poligon Frekuensi')
ax.fill_between(tt_extended, fr_extended, alpha=0.15, color='steelblue')

# Tandai titik tengah
for xi, fi in zip(titik_tengah, freq):
    ax.annotate(f'({xi:.1f}, {fi})', (xi, fi),
                textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=8, color='navy')

ax.set_xlabel('Titik Tengah Kelas (%)')
ax.set_ylabel('Frekuensi (Jumlah Provinsi)')
ax.set_title('Poligon Frekuensi: Distribusi Persentase Lulusan SMA+\n38 Provinsi di Indonesia (2025)')
ax.legend()

plt.tight_layout()
plt.savefig('outputs/figures/02_poligon_frekuensi.png', bbox_inches='tight')
plt.show()
print('Tersimpan: outputs/figures/02_poligon_frekuensi.png')

## Grafik 3: Ogif Positif (Frekuensi Kumulatif Kurang Dari)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Ogif: titik (tepi_atas[i], fk_kurang[i]) + titik awal (tepi_bawah[0], 0)
x_ogif_pos = [tepi_bawah[0]] + list(tepi_atas)
y_ogif_pos = [0] + list(fk_kurang)

ax.plot(x_ogif_pos, y_ogif_pos, 's-', color='green',
        linewidth=2, markersize=8, markerfacecolor='lightgreen', label='Ogif Positif')

# Anotasi tiap titik
for xi, yi in zip(x_ogif_pos[1:], y_ogif_pos[1:]):
    ax.annotate(f'{yi}', (xi, yi),
                textcoords='offset points', xytext=(5, 5),
                fontsize=9, color='darkgreen')

# Garis bantu n/2 untuk median
ax.axhline(y=n/2, color='red', linestyle='--', alpha=0.6, label=f'n/2 = {n/2} (letak median)')

ax.set_xlabel('Tepi Atas Kelas (%)')
ax.set_ylabel('Frekuensi Kumulatif Kurang Dari')
ax.set_title('Ogif Positif (Kurang Dari): Distribusi Lulusan SMA+\n38 Provinsi di Indonesia (2025)')
ax.legend()
ax.set_ylim(0, n + 2)

plt.tight_layout()
plt.savefig('outputs/figures/03_ogif_positif.png', bbox_inches='tight')
plt.show()
print('Tersimpan: outputs/figures/03_ogif_positif.png')

## Grafik 4: Ogif Negatif (Frekuensi Kumulatif Lebih Dari)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Ogif negatif: titik (tepi_bawah[i], fk_lebih[i]) + titik akhir (tepi_atas[-1], 0)
x_ogif_neg = list(tepi_bawah) + [tepi_atas[-1]]
y_ogif_neg = list(fk_lebih) + [0]

ax.plot(x_ogif_neg, y_ogif_neg, 'D-', color='orangered',
        linewidth=2, markersize=8, markerfacecolor='lightsalmon', label='Ogif Negatif')

for xi, yi in zip(x_ogif_neg[:-1], y_ogif_neg[:-1]):
    ax.annotate(f'{yi}', (xi, yi),
                textcoords='offset points', xytext=(5, 5),
                fontsize=9, color='darkred')

ax.axhline(y=n/2, color='blue', linestyle='--', alpha=0.6, label=f'n/2 = {n/2}')

ax.set_xlabel('Tepi Bawah Kelas (%)')
ax.set_ylabel('Frekuensi Kumulatif Lebih Dari')
ax.set_title('Ogif Negatif (Lebih Dari): Distribusi Lulusan SMA+\n38 Provinsi di Indonesia (2025)')
ax.legend()
ax.set_ylim(0, n + 2)

plt.tight_layout()
plt.savefig('outputs/figures/04_ogif_negatif.png', bbox_inches='tight')
plt.show()
print('Tersimpan: outputs/figures/04_ogif_negatif.png')

## Grafik 5: Ogif Positif & Negatif Overlay (X-Banner Hero)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(x_ogif_pos, y_ogif_pos, 's-', color='green',
        linewidth=2.5, markersize=8, label='Ogif Positif (Kurang Dari)')
ax.plot(x_ogif_neg, y_ogif_neg, 'D-', color='orangered',
        linewidth=2.5, markersize=8, label='Ogif Negatif (Lebih Dari)')
ax.axhline(y=n/2, color='gray', linestyle=':', alpha=0.8, label=f'n/2 = {n/2}')

ax.set_xlabel('Persentase Lulusan SMA+ (%)')
ax.set_ylabel('Frekuensi Kumulatif')
ax.set_title('Ogif Positif & Negatif: Distribusi Lulusan SMA+\n38 Provinsi Indonesia (2025)')
ax.legend()

plt.tight_layout()
plt.savefig('outputs/figures/05_ogif_overlay.png', bbox_inches='tight')
plt.show()
print('Tersimpan: outputs/figures/05_ogif_overlay.png')

## Grafik 6: Scatter Plot X vs Y (Preview untuk Korelasi)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(X, Y, c=Y, cmap='RdYlGn_r', s=80, edgecolors='navy', linewidth=0.5, zorder=5)
plt.colorbar(scatter, ax=ax, label='TPT (%)')

# Label tiap titik
for i, prov in enumerate(df['Provinsi']):
    ax.annotate(prov, (X[i], Y[i]),
                fontsize=6, alpha=0.7,
                textcoords='offset points', xytext=(3, 3))

ax.set_xlabel('Persentase Lulusan SMA+ per Provinsi (%)')
ax.set_ylabel('Tingkat Pengangguran Terbuka (TPT) Agustus 2025 (%)')
ax.set_title('Scatter Plot: Hubungan Pendidikan SMA+ vs TPT\n38 Provinsi Indonesia (2025)')

plt.tight_layout()
plt.savefig('outputs/figures/06_scatter_preview.png', bbox_inches='tight')
plt.show()
print('Tersimpan: outputs/figures/06_scatter_preview.png')